# Séance 6 - Explorer les collections numérisées : IIIF

Master 2 TNAH, École nationale des chartes

Dans ce notebook, on suit le chemin d'une image, du manifeste jusqu'au fichier enregistré sur l'ordinateur :

1. [Parcourir les manifestes](#1.-Parcourir-les-manifestes)
2. [Construire des URL IIIF](#2.-Construire-des-URL-IIIF)
3. [Télécharger avec `requests`](#3.-Télécharger-avec-requests)
4. [Télécharger et enregistrer une image](#4.-Télécharger-et-enregistrer-une-image)
5. [Traiter l'image avec Pillow](#5.-Traiter-l'image-avec-Pillow)
6. [Exercice guidé : télécharger plusieurs vues de Berne](#6.-Exercice-guidé-:-télécharger-plusieurs-vues-de-Berne)

On travaille sur deux manuscrits du corpus :

- **Berne, Burgerbibliothek, Cod. 318** : le *Physiologus Bernensis*, diffusé par e-codices (manifeste IIIF **v2**) ;
- **Londres, British Library, Add MS 11283** : un bestiaire, diffusé par la British Library (manifeste IIIF **v3**).

## 0. Mise en place

`requests` et Pillow ne sont pas des modules natifs : il faut les installer une fois. Dans un terminal :

```bash
pip install requests pillow
```

Sur Google Colab, on écrit la même commande dans une cellule, précédée d'un point d'exclamation : `!pip install requests pillow`.

Attention : on **installe** `pillow`, mais on **importe** `PIL`.

Les deux manifestes doivent se trouver dans le dossier `donnees`, à côté de ce notebook : `donnees/Berne_318.json` et `donnees/Add_11283.json`.

In [ ]:
import json
from pathlib import Path

import requests
from PIL import Image

## 1. Parcourir les manifestes

### 1.1 Charger les deux manifestes

On commence avec les copies locales, comme en séance 5.

In [ ]:
with open(Path("donnees") / "Berne_318.json", encoding="utf-8") as fichier:
    manifeste_berne = json.load(fichier)

with open(Path("donnees") / "Add_11283.json", encoding="utf-8") as fichier:
    manifeste_londres = json.load(fichier)

print(type(manifeste_berne), type(manifeste_londres))

### 1.2 Les clés en `@` et la version du manifeste

Avant d'écrire du code, on lit `@context` : il indique la version de l'API Présentation.

In [ ]:
print(manifeste_berne["@context"])
print(manifeste_londres["@context"])

En v2, l'identifiant et le type sont des mots-clés JSON-LD, avec un `@`. En v3, seul `@context` garde son `@`.

In [ ]:
print(manifeste_berne["@id"])
print(manifeste_berne["@type"])

print(manifeste_londres["id"])
print(manifeste_londres["type"])

Le titre (`label`) est une chaîne en v2, mais un **dictionnaire de langues** en v3. Chaque valeur de ce dictionnaire est une liste.

In [ ]:
print(manifeste_berne["label"])
print(manifeste_londres["label"])
print(manifeste_londres["label"]["en"][0])

#### Exercice 1

Pour chacun des deux manifestes, affichez toutes les métadonnées (`metadata`) sous la forme `étiquette : valeur`, une par ligne.

*Indice :* en v2, chaque entrée est un dictionnaire `{"label": ..., "value": ...}` dont les valeurs sont des chaînes. En v3, l'étiquette et la valeur sont à leur tour des dictionnaires de langues.

Que constatez-vous en comparant les deux listes ?

In [ ]:
# Votre code ici

### 1.3 Les canvas : des vues, pas des folios

En v2, les vues sont dans `sequences`, une liste dont on prend le premier élément, puis dans `canvases`. En v3, elles sont directement dans `items`.

In [ ]:
canvases = manifeste_berne["sequences"][0]["canvases"]
print(len(canvases))

for canvas in canvases[:6]:
    print(canvas["label"])

In [ ]:
vues_londres = manifeste_londres["items"]
print(len(vues_londres))

for canvas in vues_londres[:6]:
    print(canvas["label"]["en"][0])

Berne compte 131 feuillets mais 281 vues : les plats de reliure, les gardes, la tranche, les règles et les mires de couleur sont aussi numérisés.

#### Exercice 2

Trouvez, avec une boucle, l'**indice** du f. 1r dans chacun des deux manifestes. Les labels ne s'écrivent pas de la même façon : `"1r"` à Berne, `"f.1r"` à Londres. Rangez les résultats dans `indice_berne` et `indice_londres`.

*Indice :* `enumerate()` donne, à chaque tour de boucle, l'indice et l'élément :

```python
for indice, creature in enumerate(["dragon", "licorne", "lion"]):
    print(indice, creature)
```

In [ ]:
# Votre code ici



print(indice_berne, indice_londres)

### 1.4 Descendre jusqu'à l'image

L'image est rattachée au canvas par une **annotation**. En descendant, on trouve deux adresses :

- **l'URL de l'image**, à télécharger telle quelle ;
- **la base de l'URL**, qui permet de demander une autre région ou une autre taille (partie 2).

In [ ]:
# v2 : Berne
image = canvases[4]["images"][0]["resource"]
print(image["@id"])
print(image["service"]["@id"])

In [ ]:
# v3 : Londres
image = vues_londres[9]["items"][0]["items"][0]["body"]
print(image["id"])
print(image["service"][1]["id"])

Deux remarques sur ces adresses :

- L'URL de l'image de Berne se termine par `default/jpg` au lieu de `default.jpg` : c'est une coquille d'e-codices, présente sur les 281 vues. D'où l'intérêt de reconstruire l'URL à partir de la base.
- La British Library propose **deux** services pour chaque image, un par version de l'API Image. On prend le second, `[1]`, celui de l'API Image 3. Le premier utilise la clé `"@id"` et non `"id"`.

#### Exercice 3

Récupérez la base de l'URL :

- du f. 2r de Berne, dans `base_berne_2r` ;
- du f. 23r de Londres (le paon et la poule), dans `base_londres_23r`.

Parcourez les canvas avec une boucle, comme dans l'exercice 2 : cette fois, on n'a pas besoin de l'indice.

In [ ]:
# Votre code ici



print(base_berne_2r)
print(base_londres_23r)

## 2. Construire des URL IIIF

Une URL d'image suit le schéma :

`{scheme}://{server}{/prefix}/{identifier}/{region}/{size}/{rotation}/{quality}.{format}`

La base de l'URL (les quatre premiers éléments) vient du manifeste. On y ajoute les cinq paramètres avec une f-string.

Attention à la version de l'API Image : Berne utilise la **version 2**, où la taille maximale s'écrit `full` ; Londres utilise la **version 3**, où elle s'écrit `max`.

In [ ]:
base_berne = canvases[4]["images"][0]["resource"]["service"]["@id"]
base_londres = vues_londres[9]["items"][0]["items"][0]["body"]["service"][1]["id"]

url_berne = f"{base_berne}/full/full/0/default.jpg"
url_londres = f"{base_londres}/full/max/0/default.jpg"
url_info = f"{base_londres}/info.json"

print(url_berne)
print(url_londres)
print(url_info)

Copiez ces adresses dans votre navigateur : la première affiche le f. 1r de Berne, la deuxième le f. 1r de Londres, la troisième la fiche technique de l'image de Londres, en JSON.

#### Exercice 4

Construisez :

1. l'URL du f. 1r de Londres en **niveaux de gris**, réduit à **800 pixels de large** (`url_gris`) ;
2. l'URL du f. 1r de Berne entier, tourné de **90°** (`url_tournee`) ;
3. l'URL d'un détail du f. 1r de Londres : le rectangle qui commence à 1400 pixels du bord gauche et 3000 pixels du haut, large de 2000 pixels et haut de 1500 (`url_detail`).

Vérifiez chaque URL dans le navigateur.

In [ ]:
# Votre code ici

## 3. Télécharger avec `requests`

### 3.1 Envoyer une requête

On télécharge cette fois le manifeste de Berne depuis le Web, à l'adresse indiquée dans son `@id`.

In [ ]:
url_manifeste = manifeste_berne["@id"]
reponse = requests.get(url_manifeste, timeout=10)

print(reponse.status_code)
print(reponse.headers["Content-Type"])

In [ ]:
manifeste_en_ligne = reponse.json()

print(manifeste_en_ligne["label"])
print(len(manifeste_en_ligne["sequences"][0]["canvases"]))

`reponse.json()` donne le même dictionnaire que `json.load()` sur la copie locale : seule l'origine change.

### 3.2 Vérifier la réponse

Première façon : tester le code de retour avec une condition.

In [ ]:
reponse = requests.get(url_manifeste, timeout=10)

if reponse.status_code == 200:
    manifeste_en_ligne = reponse.json()
    print("Manifeste téléchargé.")
else:
    print(f"Échec du téléchargement : code {reponse.status_code}")

Seconde façon : `raise_for_status()` lève une exception si le code indique une erreur, et on la gère avec `try` et `except`.

In [ ]:
try:
    reponse = requests.get(url_manifeste, timeout=10)
    reponse.raise_for_status()
    manifeste_en_ligne = reponse.json()
    print("Manifeste téléchargé.")
except requests.exceptions.RequestException as erreur:
    print(f"Erreur lors du téléchargement : {erreur}")

#### Exercice 5

Reprenez le bloc `try` / `except` ci-dessus avec une URL fautive : remplacez `bbb-0318` par `bbb-9999` dans l'adresse du manifeste. Quel message obtenez-vous ? Quel est le code de retour ?

*Indice :* la méthode `replace()` des chaînes de caractères remplace un morceau de texte par un autre.

In [ ]:
# Votre code ici

## 4. Télécharger et enregistrer une image

### 4.1 Télécharger le f. 1r de Londres

Une image n'est pas du JSON : on utilise `reponse.content`, le contenu brut, en *bytes*. On vérifie d'abord le type du contenu avec `in`, car le serveur ajoute parfois des précisions (`; charset=UTF-8`).

In [ ]:
reponse = requests.get(url_londres, timeout=30)
reponse.raise_for_status()

print(reponse.headers["Content-Type"])
print("image/jpeg" in reponse.headers["Content-Type"])
print(type(reponse.content))

### 4.2 Enregistrer l'image

On crée le dossier `images` avec `pathlib`, puis on écrit le fichier en mode `"wb"` : `"w"` pour écrire, `"b"` pour du binaire.

In [ ]:
dossier = Path("images")
dossier.mkdir(exist_ok=True)
chemin = dossier / "add_ms_11283_f1r.jpg"

with open(chemin, "wb") as fichier:
    fichier.write(reponse.content)

print(chemin.exists())

#### Exercice 6

Téléchargez le f. 1r de Berne à **1000 pixels de large**, vérifiez qu'il s'agit bien d'une image JPEG, puis enregistrez-le dans `images/berne_318_f1r.jpg`. Si le contenu n'est pas une image JPEG, affichez un message à la place.

In [ ]:
# Votre code ici

## 5. Traiter l'image avec Pillow

### 5.1 Ouvrir l'image

`Image.open()` renvoie un **objet** image, avec des attributs et des méthodes. Pour l'afficher dans le notebook, on écrit simplement `image` en dernière ligne de la cellule (`image.show()` ouvrirait un programme externe).

In [ ]:
image = Image.open(Path("images") / "add_ms_11283_f1r.jpg")

print(image.size)
print(image.format)
print(image.mode)

In [ ]:
image

### 5.2 Transformer l'image

Chaque méthode renvoie une **nouvelle** image : l'original n'est pas modifié.

Attention aux conventions, qui ne sont pas celles de l'URL IIIF :

- `crop()` attend les coordonnées des coins (gauche, haut, droite, bas), et non une largeur et une hauteur ;
- `resize()` ne garde pas les proportions : il faut calculer la hauteur ;
- `rotate()` tourne dans le sens **inverse** des aiguilles d'une montre et rogne les coins sans `expand=True`.

In [ ]:
# Recadrer : la même région que url_detail (x, y, largeur, hauteur)
x, y, l, h = 1400, 3000, 2000, 1500
detail = image.crop((x, y, x + l, y + h))
print(detail.size)

In [ ]:
# Redimensionner à 800 pixels de large, en gardant les proportions
largeur, hauteur = image.size
reduite = image.resize((800, round(hauteur * 800 / largeur)))
print(reduite.size)

In [ ]:
# Tourner de 90° dans le sens des aiguilles d'une montre, comme la rotation IIIF
tournee = image.rotate(-90, expand=True)
print(tournee.size)

In [ ]:
# Enregistrer le détail au format PNG
detail.save(Path("images") / "add_ms_11283_f1r_detail.png")
detail

### 5.3 Comparer avec l'URL IIIF

On télécharge le même détail, recadré cette fois par le serveur : on construit l'URL avec les mêmes variables `x`, `y`, `l` et `h`, puis on compare les deux images.

In [ ]:
url_detail = f"{base_londres}/{x},{y},{l},{h}/max/0/default.jpg"
reponse = requests.get(url_detail, timeout=30)
reponse.raise_for_status()

chemin_detail_iiif = Path("images") / "add_ms_11283_f1r_detail_iiif.jpg"
with open(chemin_detail_iiif, "wb") as fichier:
    fichier.write(reponse.content)

detail_iiif = Image.open(chemin_detail_iiif)
print(detail.size, detail_iiif.size)

#### Exercice 7

Ouvrez l'image de Berne enregistrée dans l'exercice 6, puis :

1. redimensionnez-la à **800 pixels de large** en conservant les proportions ;
2. convertissez le résultat en niveaux de gris avec la méthode `convert("L")` ;
3. enregistrez-le dans `images/berne_318_f1r_gris.jpg` et affichez-le.

In [ ]:
# Votre code ici

## 6. Exercice guidé : télécharger plusieurs vues de Berne

On assemble toutes les étapes de la séance pour télécharger les **cinq premières vues du texte** de Berne (du f. 1r au f. 2ar), et garder une notice pour chacune.

On demande les images à **800 pixels de large** : télécharger les 281 vues en pleine résolution prendrait plus d'un gigaoctet, beaucoup de temps, et le serveur finirait par répondre `429` (*Too Many Requests*).

### Étape 1 : parcourir les canvas

Pour chaque vue, on range dans un dictionnaire son indice, son label, la base de l'URL et ses dimensions déclarées dans le manifeste. On obtient une liste de dictionnaires, `notices`.

In [ ]:
notices = []

for indice, canvas in enumerate(canvases):
    if 4 <= indice <= 8:
        service = canvas["images"][0]["resource"]["service"]
        notices.append({
            "indice": indice,
            "label": canvas["label"],
            "base": service["@id"],
            "largeur_declaree": canvas["width"],
            "hauteur_declaree": canvas["height"],
        })

print(len(notices))
print(notices[0])

**Explications**

- `enumerate()` donne l'indice de chaque canvas ; on garde ceux dont l'indice va de 4 à 8.
- L'indice est utile quand les labels ne sont pas fiables : certains manifestes donnent le même label à toutes les vues (souvent `NP`, *non paginé*).
- Les dimensions déclarées sont celles de l'image en pleine résolution.

### Étape 2 : télécharger et enregistrer les images

On crée un dossier pour le manuscrit, puis on télécharge chaque vue. On ajoute à chaque notice le code HTTP et le nom du fichier.

In [ ]:
dossier_berne = Path("images") / "berne_318"
dossier_berne.mkdir(parents=True, exist_ok=True)

for notice in notices:
    url = f"{notice['base']}/full/800,/0/default.jpg"
    try:
        reponse = requests.get(url, timeout=30)
        notice["code_http"] = reponse.status_code
        reponse.raise_for_status()

        if "image/jpeg" in reponse.headers["Content-Type"]:
            chemin = dossier_berne / f"berne_318_{notice['label']}.jpg"
            with open(chemin, "wb") as fichier:
                fichier.write(reponse.content)
            notice["fichier"] = chemin.name
            print(f"{notice['label']} : téléchargé ({reponse.status_code})")
        else:
            print(f"{notice['label']} : ce n'est pas une image JPEG")
    except requests.exceptions.RequestException as erreur:
        print(f"{notice['label']} : erreur ({erreur})")

**Explications**

- `mkdir(parents=True, exist_ok=True)` crée `images/berne_318`, et `images` au passage s'il n'existe pas encore.
- Le code HTTP est enregistré **avant** `raise_for_status()`, pour garder une trace même en cas d'échec.
- La clé `"fichier"` n'est ajoutée que si l'image a bien été enregistrée.

### Étape 3 : relever les dimensions réelles avec Pillow

On ouvre chaque image enregistrée et on ajoute à la notice ses dimensions réelles.

In [ ]:
for notice in notices:
    if "fichier" in notice:
        image = Image.open(dossier_berne / notice["fichier"])
        notice["largeur_telechargee"], notice["hauteur_telechargee"] = image.size

print(notices[0])

**Explications**

- `"fichier" in notice` teste si la clé existe : on ne traite que les images téléchargées.
- `image.size` renvoie un tuple `(largeur, hauteur)`, qu'on range directement dans deux clés.
- On peut comparer les dimensions déclarées et téléchargées : le rapport largeur / hauteur doit être le même.

### Étape 4 : enregistrer les notices en JSON

On enregistre la liste `notices` dans un fichier JSON, comme en séance 5.

In [ ]:
with open(dossier_berne / "notices.json", "w", encoding="utf-8") as fichier:
    json.dump(notices, fichier, indent=4, ensure_ascii=False)

print((dossier_berne / "notices.json").exists())

### Étape 5 : automatiser avec une fonction

Écrivez une fonction `telecharger_vues(manifeste, debut, fin, dossier)` qui reprend les étapes 1 à 4 pour n'importe quel manifeste **v2** :

- `manifeste` : le manifeste, déjà chargé (un dictionnaire) ;
- `debut` et `fin` : les indices de la première et de la dernière vue à télécharger ;
- `dossier` : le chemin du dossier où enregistrer les images et `notices.json`.

La fonction renvoie la liste des notices. Testez-la sur les vues 9 à 12 de Berne (du f. 2av au f. 4r), dans le dossier `images/berne_318_suite`.

N'oubliez pas d'appeler la fonction !

In [ ]:
# Votre code ici